In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3.5:latest",
    temperature=0.5,
    max_tokens=250
)

tc_llm = ChatOllama(
    base_url="http://localhost:11434",
    model="gemma4:latest",
    temperature=0.5,
    max_tokens=250
)

In [2]:
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate


@tool
def generate_test_cases(user_story: str) -> str:
    "Generates test case from the user stories"
    prompt_template = PromptTemplate.from_template(

        """
        You are are QA Automation Engineer.
        Your task is to convert the following user stories into at least 10 testcases, in Gherkin BDD style format.
        Include combinations of valid, invalid, edge case and alternative flow scearios

        User Stories: {user_story}

        Format:
        Feature: ...
        Scenario: ...
            Given ...
            When ...
            Then ...

        """
    )
    prompt = prompt_template.invoke({"user_story": user_story})
    return tc_llm.invoke(prompt).content


In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Creating an AI Agent (replaces deprecated initialize_agent + AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)
agent = create_agent(
    tools=[generate_test_cases],
    model=llm,
)

user_story_input = """
    As a user, I want to reset my password using a link sent to my email, so that I can
    regain access to my account if I forgot my password.
"""

result = agent.invoke(
    {"messages": [HumanMessage(content=f"Create BDD Test cases for: {user_story_input}")]}
)

print(result["messages"][-1].content)


Here are the comprehensive BDD Test Cases for the Password Reset feature:

## Feature: Password Reset via Email Link

This feature validates the complete process of a user recovering access to their account by receiving and utilizing a unique, time-sensitive link sent to their registered email address.

---

### 🟢 Valid & Happy Path Scenarios

**Scenario 1: Successful Password Reset and Account Access**
*(The primary, successful flow)*
*   **Given** a registered user exists with a forgotten password
*   **And** the user initiates the "Forgot Password" flow
*   **When** the system sends a valid, unique reset link to the user's email
*   **And** the user clicks the link and successfully sets a new, compliant password
*   **Then** the user should be redirected to a confirmation page
*   **And** the user should be able to log in using the new password

**Scenario 2: Password Complexity Validation (Successful Update)**
*(Validating the constraints of the new password)*
*   **Given** the use